# datamax-dpl quickstart

Describe a label in JSON, check it against a Datamax printer model, look at the DPL it becomes and
draw a preview. Nothing here needs a printer: the job is written to a file.

No real printer has been tested with datamax-dpl yet; the output follows the 1998 DPL manual.

## 1) Install

Inside a checkout of the repository the cell installs that checkout; anywhere else it installs
from GitHub. It installs into this kernel's Python, so the package can be imported right away.

In [ ]:
# IPython sets _exit_code after every ! line; the asserts make a failing command fail the notebook.
import sys
from pathlib import Path


def checkout_root() -> Path | None:
    """The datamax-dpl checkout this notebook runs in, if any."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        pyproject = folder / "pyproject.toml"
        if pyproject.is_file() and 'name = "datamax-dpl"' in pyproject.read_text(encoding="utf-8"):
            return folder
    return None


root = checkout_root()
source = str(root) if root is not None else "git+https://github.com/bitranox/datamax-dpl"
!{sys.executable} -m pip install --quiet {source}
assert _exit_code == 0
source

## 2) The command line

`python -m datamax_dpl` is the same CLI as the `datamax-dpl` command.

In [ ]:
!{sys.executable} -m datamax_dpl --version
assert _exit_code == 0
!{sys.executable} -m datamax_dpl models
assert _exit_code == 0

## 3) A label

Lengths are millimetres. `x` runs from the left edge, `y` from the BOTTOM edge, and every element
is placed by its lower-left corner (a circle by its centre).

In [ ]:
import json
import tempfile

work = Path(tempfile.mkdtemp(prefix="datamax-dpl-quickstart-"))
label = {
    "unit": "mm",
    "width": 60,
    "height": 30,
    "elements": [
        {"type": "text", "x": 3, "y": 21, "text": "ART. 4711", "font": 3},
        {"type": "barcode", "x": 3, "y": 3, "symbology": "ean13", "data": "400638133393", "height": 12},
        {"type": "circle", "x": 52, "y": 22, "radius": 4, "fill": 1},
    ],
}
label_file = work / "label.json"
label_file.write_text(json.dumps(label, indent=2), encoding="utf-8")
label_file

## 4) Validate it for two models

A DMX 600 draws circles itself. A Prodigy cannot, so the circle becomes a downloaded bitmap and
the result reports it as a warning. With `--policy strict` the same finding would be an error.

In [ ]:
!{sys.executable} -m datamax_dpl validate {label_file} --model dmx-600
assert _exit_code == 0
!{sys.executable} -m datamax_dpl validate {label_file} --model prodigy
assert _exit_code == 0

## 5) Render from Python and read the DPL

In [ ]:
from datamax_dpl.domain.label import Label
from datamax_dpl.domain.render import build_job
from datamax_dpl.domain.render_context import RenderOptions

result = build_job(Label.model_validate(label), RenderOptions(model="dmx-600"))
result.ok, result.record_count

In [ ]:
from IPython.display import Pretty

Pretty(result.data.replace(b"\x02", b"<STX>").replace(b"\r", b"<CR>\n").decode("ascii"))

In [ ]:
[(finding.severity.value, finding.code, finding.message) for finding in result.findings]

## 6) JSON for scripts and LLMs

`--json-bare` answers with one line: `{ok, command, data, skipped}`. Exit codes: 0 done, 1 refused,
2 could not run.

In [ ]:
answer_file = work / "answer.json"
!{sys.executable} -m datamax_dpl --json-bare validate {label_file} --model prodigy --policy strict > {answer_file}
assert _exit_code == 1
answer = json.loads(answer_file.read_text(encoding="utf-8"))
answer["ok"], [(f["code"], f["severity"]) for f in answer["data"]["findings"]]

## 7) Preview and "print" to a file

The preview is drawn at the printer's resolution. Text shows as grey cells unless the `preview`
extra (Pillow) is installed.

In [ ]:
from IPython.display import Image

!{sys.executable} -m datamax_dpl preview {label_file} --model dmx-600 -o {work / "label.png"}
assert _exit_code == 0
Image(filename=str(work / "label.png"))

In [ ]:
!{sys.executable} -m datamax_dpl print {label_file} --model dmx-600 --printer file:{work / "job.dpl"}
assert _exit_code == 0
(work / "job.dpl").stat().st_size